# Phase 22: Learning-Curve Diagnostics

## Mục đích

Phân tích learning dynamics của LSTM B0 và Transformer B0 từ training histories.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from course_work.diagnostics.learning_curves import LearningCurveDiagnostics

## Step 1: Load Source Data

In [ ]:
# Initialize diagnostics
artifacts_dir = Path("../../artifacts")
diagnostics = LearningCurveDiagnostics(artifacts_dir)

# Load source data
load_results = diagnostics.load_source_data()
print(json.dumps(load_results, indent=2))

## Step 2: Validate History Integrity

In [ ]:
# Validate LSTM history
if diagnostics.lstm_history is not None:
    lstm_integrity = diagnostics.validate_history_integrity(
        diagnostics.lstm_history, 
        diagnostics.LSTM_RUN_ID
    )
    print("LSTM Integrity:", lstm_integrity["status"])

# Validate Transformer history
if diagnostics.transformer_history is not None:
    tf_integrity = diagnostics.validate_history_integrity(
        diagnostics.transformer_history, 
        diagnostics.TRANSFORMER_RUN_ID
    )
    print("Transformer Integrity:", tf_integrity["status"])

## Step 3: Verify Fairness Gates

In [ ]:
fairness = diagnostics.verify_fairness()
print(json.dumps(fairness, indent=2))

## Step 4: LSTM Diagnostics

In [ ]:
if diagnostics.lstm_history is not None:
    lstm_best_epoch, lstm_best_rmse = diagnostics.recompute_best_epoch(diagnostics.lstm_history)
    lstm_initial = diagnostics.compute_initial_diagnostics(diagnostics.lstm_history, "LSTM")
    lstm_tail = diagnostics.compute_tail_diagnostics(diagnostics.lstm_history, "LSTM", lstm_best_epoch)
    lstm_grad = diagnostics.compute_gradient_diagnostics(diagnostics.lstm_history, "LSTM")
    
    print(f"LSTM Best Epoch: {lstm_best_epoch}")
    print(f"LSTM Best RMSE: {lstm_best_rmse:.4f} Wh")
    print(f"LSTM Initial Diagnostics: {lstm_initial}")
    print(f"LSTM Gradient Diagnostics: {lstm_grad}")

## Step 5: Transformer Diagnostics

In [ ]:
if diagnostics.transformer_history is not None:
    tf_best_epoch, tf_best_rmse = diagnostics.recompute_best_epoch(diagnostics.transformer_history)
    tf_initial = diagnostics.compute_initial_diagnostics(diagnostics.transformer_history, "TRANSFORMER")
    tf_tail = diagnostics.compute_tail_diagnostics(diagnostics.transformer_history, "TRANSFORMER", tf_best_epoch)
    tf_grad = diagnostics.compute_gradient_diagnostics(diagnostics.transformer_history, "TRANSFORMER")
    
    print(f"Transformer Best Epoch: {tf_best_epoch}")
    print(f"Transformer Best RMSE: {tf_best_rmse:.4f} Wh")
    print(f"Transformer Initial Diagnostics: {tf_initial}")
    print(f"Transformer Gradient Diagnostics: {tf_grad}")

## Step 6: Model Comparison

In [ ]:
import pandas as pd

comparisons = []

if diagnostics.lstm_history is not None and diagnostics.lstm_config is not None:
    lstm_comp = diagnostics.build_model_comparison(
        diagnostics.lstm_history,
        diagnostics.LSTM_RUN_ID,
        "LSTM",
        "EARLY_STOPPING",
        diagnostics.lstm_config["model"].get("trainable_parameters", 58177)
    )
    comparisons.append(lstm_comp)

if diagnostics.transformer_history is not None and diagnostics.transformer_config is not None:
    tf_comp = diagnostics.build_model_comparison(
        diagnostics.transformer_history,
        diagnostics.TRANSFORMER_RUN_ID,
        "TRANSFORMER",
        "EARLY_STOPPING",
        None
    )
    comparisons.append(tf_comp)

comparison_df = pd.DataFrame([c.__dict__ for c in comparisons])
print(comparison_df.T)

## Step 7: Diagnostic Findings

In [ ]:
all_findings = []

if diagnostics.lstm_history is not None:
    lstm_findings = diagnostics.classify_diagnostic_findings(
        diagnostics.lstm_history,
        "LSTM",
        lstm_best_epoch,
        lstm_initial,
        lstm_tail,
        lstm_grad
    )
    all_findings.extend(lstm_findings)

if diagnostics.transformer_history is not None:
    tf_findings = diagnostics.classify_diagnostic_findings(
        diagnostics.transformer_history,
        "TRANSFORMER",
        tf_best_epoch,
        tf_initial,
        tf_tail,
        tf_grad
    )
    all_findings.extend(tf_findings)

findings_df = pd.DataFrame([
    {
        "finding_id": f.finding_id,
        "model": f.model,
        "code": f.diagnostic_code.value,
        "title": f.title,
        "confidence": f.confidence.value,
        "severity": f.severity.value,
        "action": f.action_type.value,
        "future_phase": f.mapped_future_phase
    }
    for f in all_findings
])
print(findings_df.to_string())

## Step 8: Hypothesis Registry

In [ ]:
hypotheses = diagnostics.build_hypothesis_registry(all_findings)
hypothesis_df = pd.DataFrame([
    {
        "hypothesis_id": h.hypothesis_id,
        "source_model": h.source_model,
        "statement": h.hypothesis_statement,
        "phase": h.pre_registered_phase,
        "factor": h.factor,
        "status": h.status
    }
    for h in hypotheses
])
print(hypothesis_df.to_string())

## Step 9: Diagnostic Summary

In [ ]:
summary = diagnostics.generate_diagnostic_summary()
print(json.dumps(summary, indent=2))